In [169]:
import pandas as pd
from datetime import datetime

# Load the Excel file
file_path = "./input/NHKzendingspredikanten18001960.xlsx"  # Change this to your actual file path
excel_data = pd.read_excel(file_path)

# Forward-fill missing person info to associate all places with the right person
excel_data['persoonsvermelding bij VdE'] = excel_data['persoonsvermelding bij VdE'].ffill()

# Generate integer person IDs starting from 0
unique_persons = excel_data['persoonsvermelding bij VdE'].unique()
person_ids = {name: idx for idx, name in enumerate(unique_persons)}
excel_data['person_id'] = excel_data['persoonsvermelding bij VdE'].map(person_ids)

# Create the Persons sheet
persons = (
    excel_data
    .drop(columns=['plaats', 'plaats van tot'])
    .drop_duplicates(subset='person_id')
    .reset_index(drop=True)
)

# Create the Places sheet
places = (
    excel_data[['person_id', 'plaats', 'plaats van tot']]
    .dropna(subset=['plaats'])
    .reset_index(drop=True)
)

In [170]:
# Panda settings for showing data (this is foremost done to more easily explore the data while processing it)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [171]:
persons.head()

,persoonsvermelding bij VdE,geslacht,achternaam,voorvoegsel,achtervoegsel,tussenvoegsel,initialen,voornamen,rol,leefjaren,bijzonderheden,huwelijk eerste,huwelijk tweede,verwijzing naar VdE zendelingen,person_id
0,"Adama Pzn, A., pdt",m,Adama,NaN,Pzn,NaN,A.,NaN,pdt,1844-1911+,T,NaN,NaN,NaN,0
1,"Addens, J., pdt",m,Addens,NaN,NaN,NaN,J.,NaN,pdt,- sek. 1805,T,NaN,NaN,NaN,1
2,"Adèr, J.W.H., pdt (Ev.Luth.)",m,Adèr,NaN,NaN,NaN,J.W.H.,NaN,pdt (Ev.Luth.),1830-1895,T L,NaN,NaN,NaN,2
3,"Akkerman, Roelof, gl",m,Akkerman,NaN,NaN,NaN,R.,Roelof,gl,1870- sek. 1940?,NaN,"~ C. Minderman, † Batavia 1939",NaN,NaN,3
4,"Akkerman, W., pb (zo.)",m,Akkerman,NaN,NaN,NaN,W.,NaN,pb (zo.),1887-1945,NaN,~ 1929 L. Hundhausen,NaN,NaN,4


# Spouses

In [172]:
spouses_1 = persons[['person_id', 'huwelijk eerste']].rename(columns={'huwelijk eerste': 'spouse'})
spouses_2 = persons[['person_id', 'huwelijk tweede']].rename(columns={'huwelijk tweede': 'spouse'})

spouses = pd.concat([spouses_1, spouses_2], ignore_index=True)
spouses['spouse'] = spouses['spouse'].str.replace(r'\((1|2)\)', '', regex=True)
spouses['spouse'] = spouses['spouse'].str.replace('~', '', regex=False)

In [173]:
spouses['trouwjaar'] = spouses['spouse'].str.strip().str.extract(r'^(\d{4})')

In [174]:
spouses['sterfjaar'] = spouses['spouse'].str.extract(r'†.*?(\d{4})')
spouses['naam'] = spouses['spouse'].str.replace(r'\d+', '', regex=True)
spouses['naam'] = spouses['naam'].str.replace(r'†', '', regex=True)

In [175]:
spouses = spouses.dropna(subset=['spouse'])

In [176]:
spouses.head(5)

,person_id,spouse,trouwjaar,sterfjaar,naam
3,3,"C. Minderman, † Batavia 1939",NaN,1939,"C. Minderman, Batavia"
4,4,1929 L. Hundhausen,1929,NaN,L. Hundhausen
6,6,1880 H. Pfaff † 1882,1880,1882,H. Pfaff
7,7,G. Haandrikman,NaN,NaN,G. Haandrikman
13,13,1937 M. Roelfsema,1937,NaN,M. Roelfsema


In [177]:
spouses['geboortejaar'] = ''
spouses['opmerkingen'] = ''
spouses['eind_huwelijk'] = ''

# rename id field
spouses.rename(columns={
    'person_id':'id'
}, inplace=True)

#reorder columns
spouses = spouses[['id', 'spouse', 'naam', 'geboortejaar', 'sterfjaar','trouwjaar','opmerkingen','eind_huwelijk' ]]
spouses['id'] = spouses['id'].apply(lambda x: f"phl_{x:04d}")

In [178]:
date_str = datetime.today().strftime('%m_%d_%Y')

spouses_filename = ".//output//"+ f'phl_echtgenoten_{date_str}.xlsx'
spouses.to_excel(spouses_filename, index=False)

# Persons

In [179]:
persons = persons.drop(['huwelijk eerste', 'huwelijk tweede'], axis=1)

In [180]:
persons.rename(columns={
    'person_id':'id'    
}, inplace=True)

In [181]:
persons['id'] = persons['id'].apply(lambda x: f"phl_{x:04d}")

In [182]:
persons['voornamen_of_voorletters'] = (
    persons['initialen'].fillna('').str.strip() + ', ' +
    persons['voornamen'].fillna('').str.strip()
).str.strip()

persons['voornamen_of_voorletters'] = persons['voornamen_of_voorletters'].str.replace(r',$', '', regex=True)

In [183]:
persons[['geboortejaar', 'sterfjaar']] = persons['leefjaren'].str.split('-', n=1, expand=True)

# Extract the first 4-digit number from 'geboortejaar'
persons['geboortejaar_int'] = persons['geboortejaar'].astype(str).str.extract(r'(\d{4})')
# Convert to integer (optional, depending on if you want NaN or errors on failure)
persons['geboortejaar_int'] = persons['geboortejaar_int'].astype(float).astype('Int64')
# Extract the first 4-digit number from 'sterfjaar'
persons['sterfjaar_int'] = persons['sterfjaar'].astype(str).str.extract(r'(\d{4})')
# Convert to integer (optional, depending on if you want NaN or errors on failure)
persons['sterfjaar_int'] = persons['sterfjaar_int'].astype(float).astype('Int64')

In [184]:
persons['geslacht'] = persons['geslacht'].str.upper()
persons['vde_dz']= "PHLPK"

In [185]:
persons['titel'] = (
    persons['voorvoegsel'].fillna('').str.strip() + ', ' +
    persons['achtervoegsel'].fillna('').str.strip()
).str.strip()

persons['titel'] = persons['titel'].str.replace(r',$', '', regex=True)

In [186]:
persons['titel'] = persons['titel'].str.replace(r'^,', '', regex=True)

In [187]:
rol = persons[['id','rol']] 

In [188]:
persons['titulatuur_geslacht'] = ''
persons['oorsprong'] = ''
persons['verwijzing_id'] = ''


persons.rename(columns={
    'verwijzing naar VdE zendelingen':'verwijzing',
    'bijzonderheden':'overige_details'
}, inplace=True)


persons = persons[['id', 
                   'persoonsvermelding bij VdE', #
                   'titel', #
                   'achternaam', #
                   'tussenvoegsel', # 
                   'voornamen_of_voorletters', # 
                   'titulatuur_geslacht',
                   'geslacht', #
                   'oorsprong', #
                   'verwijzing', #
                   'verwijzing_id', #
                   'geboortejaar', # 
                   'sterfjaar', #
                   'geboortejaar_int', #
                   'sterfjaar_int', #
                   'overige_details', #                   
                   'vde_dz']] #


In [189]:
rol.head()

,id,rol
0,phl_0000,pdt
1,phl_0001,pdt
2,phl_0002,pdt (Ev.Luth.)
3,phl_0003,gl
4,phl_0004,pb (zo.)


In [190]:
bio_filename = ".//output//"+ f'phl_bio_{date_str}.xlsx'
persons.to_excel(bio_filename, index=False)

# Events

In [192]:
places.rename(columns={
    'person_id':'id',
    'plaats':'werkgebied_en_soort'
}, inplace=True)
places['id'] = places['id'].apply(lambda x: f"phl_{x:04d}")

In [193]:
places[['periode_start', 'periode_eind']] = (
    places['plaats van tot']
    .astype(str)  # convert to string first
    .str.strip()
    .str.replace('–', '-', regex=False)
    .str.split('-', n=1, expand=True)
    .apply(lambda col: col.str.strip())
)

In [194]:
places['periode_start_int'] = places['periode_start'].astype(str).str.extract(r'(\d{4})')
places['periode_start_int'] = places['periode_start_int'].astype(float).astype('Int64')
places['periode_eind_int'] = places['periode_eind'].astype(str).str.extract(r'(\d{4})')
places['periode_eind_int'] = places['periode_eind_int'].astype(float).astype('Int64')

In [195]:
places.head()

,id,werkgebied_en_soort,plaats van tot,periode_start,periode_eind,periode_start_int,periode_eind_int
0,phl_0000,Banjarmasin,1892,1892,None,1892,<NA>
1,phl_0000,Madiun,1895,1895,None,1895,<NA>
2,phl_0000,Surakarta,1898,1898,None,1898,<NA>
3,phl_0000,Palembang,1901,1901,None,1901,<NA>
4,phl_0000,Padang,1903,1903,None,1903,<NA>


In [196]:
events = pd.merge(places, rol, on='id', how='left')

In [197]:
events['bijzonderheden_opmerkingen'] = ''
events['event'] = ''
events['bronvermelding'] = ''
events['details_overlijden'] = ''
events['vervolg_en_nevenrollen'] = ''
events['organisatie'] = ''

In [198]:
events = events[['id', 
                   'organisatie', #
                   'werkgebied_en_soort', #
                   'periode_start', #
                   'periode_start_int', # 
                   'periode_eind', # 
                   'periode_eind_int',
                   'bijzonderheden_opmerkingen', #
                   'event', #
                   'bronvermelding', #
                   'details_overlijden', #
                   'vervolg_en_nevenrollen']] #

In [199]:
event_filename = ".//output//"+ f'phl_event_{date_str}.xlsx'
events.to_excel(event_filename, index=False)